[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/04_surrogate_modeling.ipynb)

# 04 — Surrogate Modeling

**Purpose.** Train `f_sur: (s, theta) -> K_hat` and decide whether it is accurate
enough to optimise against. PROJECT.md section 19 Step 6.

**The question is not "is the surrogate good".** It is *"is it good enough that
an optimization result computed against it can be defended"*. Those are different
questions, and only the second one matters here — which is why section 9 is a
decision, not a summary.

**Inputs.** `data/processed/surrogate_dataset.parquet` from notebook 03.

**Outputs.** `models/surrogate.pkl` and a per-KPI error report.

## 0. Environment

Run this section first, wherever you are.

**Locally** it only walks up to the project root and makes it the working
directory, so the root-relative paths in `configs/data.yaml` resolve the same way
they do for `task clean:data` and the DVC pipeline. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path` so `import src`
works without an editable install, and installs the packages Colab does not ship.
Note that `data/` and `models/` are DVC-tracked and therefore *not* part of the
clone — a fresh runtime has neither. See the Drive cell below.

In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook. Forked the repository? Change these three values
# and the badge URL at the top of this notebook.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = ""  # the project root is the repository root

# (import name, pip name). Colab already ships numpy, pandas, pyarrow,
# scikit-learn, joblib, matplotlib and seaborn, so only these are installed —
# which keeps the bootstrap fast and avoids a "restart runtime" prompt.
COLAB_PACKAGES = [("hydra", "hydra-core")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1. Empty unless
# the Drive cell below fills it in, so local runs are unaffected.
CONFIG_OVERRIDES: list[str] = []

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ and models/ are DVC-tracked, so they are not in the Git clone and a
# fresh Colab runtime has neither. Mount Drive and point the config at it —
# Drive also survives a runtime reset, which /content does not.
#
# The scene is the large one: data/external/simulation_map/ holds 3,753 meshes,
# so keep it on Drive rather than re-downloading it per session.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# DATA_ROOT = "/content/drive/MyDrive/band-tilt/data"
# CONFIG_OVERRIDES += [
#     f"data.mdt_path={DATA_ROOT}/raw/measurement_data.csv",
#     f"data.cell_config_path={DATA_ROOT}/raw/gcell_conf.csv",
#     f"data.scene_file={DATA_ROOT}/external/simulation_map/scene.xml",
#     f"data.train_path={DATA_ROOT}/processed/mdt_train.parquet",
#     f"data.test_path={DATA_ROOT}/processed/mdt_test.parquet",
# ]

## 1. Setup

Compose the config, seed everything, and import from `src/`. Every notebook
starts the same way so that a cell copied between notebooks behaves identically.

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import load_config
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()  # matplotlib/seaborn styling for report-ready figures

pd.set_option("display.max_columns", 50)
cfg

## 2. Load D_sur and split on configurations

Whole configurations, never individual grid cells. Cells from one configuration
share the same tilts and most of the same propagation paths, so splitting within
a configuration reports an error far below the real one — and the surrogate then
looks accurate right up to the point an optimizer relies on it.

In [ ]:
from src.surrogate import dataset, features

d_sur = dataset.load(cfg)
train_df, val_df, test_df = dataset.split(d_sur, cfg)
print(f"train {len(train_df)}  val {len(val_df)}  test {len(test_df)} configurations")

## 3. Build the state and fit the feature transform

`build_state` assembles what does not vary with theta — cell geometry, band
identity, UE density — once.

`fit` is the **only** place in this project that learns from data. It sees the
training partition only: a scaler fitted on the full `D_sur` has seen the
held-out configurations, and the reported error is then optimistic for a reason
that is very hard to find later.

In [ ]:
from src.data.load import load_cell_config, load_processed
from src.data.ue_density import ue_density
from src.radio import cell_band

table = cell_band.build_table(load_cell_config(cfg), cfg)
rho = ue_density(load_processed(cfg, "train"), cfg)

state = features.build_state(table, rho, cfg)
transformer = features.fit(train_df, state, cfg)  # TRAIN ONLY

x_train = features.transform(dataset.theta_matrix(train_df), state, transformer)
x_val = features.transform(dataset.theta_matrix(val_df), state, transformer)
x_test = features.transform(dataset.theta_matrix(test_df), state, transformer)
x_train.shape

## 4. Train

Instantiated from `configs/surrogate.yaml` through Hydra's `_target_`, so
swapping architectures is a config edit and an ablation is a sweep.

If this surrogate is to feed the Bayesian Optimization loop it must expose
predictive uncertainty — an acquisition function needs a posterior, not a point
estimate.

In [ ]:
from hydra.utils import instantiate

from src.utils import tracking

kpi_cols = list(cfg.kpi.order)
y_train = train_df[kpi_cols].to_numpy()
y_val = val_df[kpi_cols].to_numpy()
y_test = test_df[kpi_cols].to_numpy()

with tracking.start_run(cfg, run_name="surrogate"):
    tracking.log_config(cfg)
    model = instantiate(cfg.surrogate)
    model.fit(x_train, y_train, x_val, y_val)

## 5. Per-KPI error

Never averaged across KPIs. A surrogate that is excellent on weak rate and
useless on hole rate is useless, because hole rate is the highest-priority
objective — and a mean over the five hides exactly that.

Report in each KPI's own units. The acceptance test is a comparison against the
improvement a result claims, and a dimensionless normalised error cannot be
compared against anything.

In [ ]:
from src.evaluation import metrics

report = metrics.prediction_error(y_test, model.predict(x_test), tuple(kpi_cols))
pd.DataFrame(report).T[["mae", "rmse", "bias", "rank_correlation"]]

## 6. Error near the optimum

**The number that actually matters.** An optimizer spends its time in the best
region of the space, so error averaged over the whole dataset is dominated by
configurations it would never propose. A surrogate can look accurate globally
while being useless exactly where it gets used.

Rank by the *true* objective, not the predicted one — ranking by prediction
selects the configurations the surrogate is most optimistic about and biases the
error downward in the very region being examined.

In [ ]:
from src.kpi import vector

true_objective = np.array([vector.scalarize(dict(zip(kpi_cols, row, strict=True)), cfg)
                           for row in y_test])
near = metrics.error_near_optimum(y_test, model.predict(x_test), true_objective,
                                  tuple(kpi_cols), quantile=0.1)
pd.DataFrame(near).T[["mae", "rmse", "bias"]]

## 7. Ranking fidelity

Both optimizers use the surrogate to *choose between* configurations. A model
with a constant bias but the right ordering optimises perfectly; an unbiased one
that shuffles the ranking does not. Weight this more heavily than MAE when
deciding whether the surrogate is good enough.

In [ ]:
pred = model.predict(x_test)
fig, axes = plt.subplots(1, len(kpi_cols), figsize=(16, 3.2))
for ax, (i, col) in zip(axes, enumerate(kpi_cols), strict=True):
    ax.scatter(y_test[:, i], pred[:, i], s=12, alpha=0.6)
    lims = [min(y_test[:, i].min(), pred[:, i].min()), max(y_test[:, i].max(), pred[:, i].max())]
    ax.plot(lims, lims, ls="--", lw=1)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("true")
axes[0].set_ylabel("predicted")
plt.tight_layout()

## 8. Acceptance decision

Compare each per-KPI error against `cfg.surrogate.acceptance.max_mae`, and
against the size of the improvement the optimization is expected to claim.

**If predicted hole rate is off by more than the improvement being claimed, the
result is noise.** That is the test. Record the decision here — a surrogate that
quietly ships below tolerance produces optimization results nobody can defend,
and the failure surfaces much later and far more expensively.

In [ ]:
acceptance = pd.DataFrame(
    {
        "mae": {k: report[k]["mae"] for k in kpi_cols},
        "max_mae": dict(cfg.surrogate.acceptance.max_mae),
    }
)
acceptance["passes"] = acceptance["mae"] <= acceptance["max_mae"]
acceptance

# TODO: state the expected improvement per KPI, and check the MAE is smaller
# than it. Do not proceed to notebooks 05a/05b until this holds or the gap is
# recorded as a known limitation of every result that follows.

## 9. Persist

The transformer travels with the model. Saved apart, the reloaded surrogate
receives tilts on a different scale from the one it was trained on and
mispredicts silently.

The cell-band ordering travels too — a theta vector cannot be interpreted without
the column order it was built with.

In [ ]:
model.transformer = transformer
model.save(cfg.surrogate.artifact_path)
print(f"wrote {cfg.surrogate.artifact_path}")

## 10. Handoff checklist

- [ ] The feature transform was fitted on the training partition only.
- [ ] The split held out whole configurations, not grid cells.
- [ ] Per-KPI errors are reported in each KPI's own units.
- [ ] Near-optimum error (section 6) is reported, not just the global error.
- [ ] The acceptance decision in section 8 is recorded, pass or fail.
- [ ] The artifact carries the transformer, the cell-band ordering and the KPI names.
- [ ] If BO will use this surrogate, it exposes predictive uncertainty.